# TRTR (Train Synthetic, Test Real)

# 1. Installing libraries

In [ ]:
!pip install -q --upgrade gdown

In [ ]:
!pip install uv

In [ ]:
!uv pip install torch pytorch_lightning xgboost optuna lightgbm catboost tqdm

# 2. Importing libraries and taking configurations

In [ ]:
import os
import json
import warnings
import time
import resource
from datetime import datetime

import numpy as np
import pandas as pd

import gdown
import optuna
from google.colab import drive

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import torch
import pytorch_lightning as pl

from tqdm import tqdm

Ensuring all scikit learn algorithms run with GPUs

In [ ]:
%load_ext cuml.accel

Connecting to google drive

In [ ]:
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/TSTR_CTGAN_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
np.random.seed(42); torch.manual_seed(42)

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__} | PL {pl.__version__} | {DEVICE}")

SEEDS = [42, 137, 256, 1024, 7, 314, 999, 2048, 55, 8080,
         101, 202, 303, 404, 505, 606, 707, 808, 909, 1111]

N_TRIALS = 40

In [ ]:
_WALL_CLOCK = {}   # "model__dataset__mode" → seconds
_PEAK_MEM   = {}   # same key → ΔRSS in MB

def _rss_mb():
    """Current RSS in MB (Linux)."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

def _snap(label, t0, m0):
    _WALL_CLOCK[label] = time.time() - t0
    _PEAK_MEM[label]   = _rss_mb() - m0

#3. Importing wells data

In [ ]:
import gdown
from tqdm import tqdm

file_ids = [
    # Synthetic Zone 1
    '1qCf7hgYtcD8jKJOM2wJHK5IxU1nlhqCQfCTF8mMAq38',
    '1TUi7a-H6bXbTtzz8gqmegShxQ6l9aewr2N0wc5fBoUs',
    '1VJn6Q6r0im_-MCWoGAyg_GIxWXbf1cAcEq5PXF-MfHw',
    # Synthetic Zone 2
    '1-7KnWpetE1MNpawwT7j5yUsBnknUCG5arS4k4MPd6ak',
    '1rcx48slhmKkll6H4Tz7qegvYghtV8qoL22cpLjhtS2k',
    '1hWQ3QCFCNPCWtX8DjmE0DEPJf-Hirg_VQt5oqsfOKqw',
    # Synthetic Zone 3
    '1iiWp8LcmbflZ-f24PTMllFqdmL7SYPRJ8ivRDTYENRw',
    '12L-Ho2Lc6SWnfOtg3yvstU88GcI-A87vTGHI3OSWtY4',
    '13Ihz76KY3dBwy5i-6RX70xqqLT0iDgWDtF4d_KFbyVc',
    # Real Zone 1
    '1opPeaM4t-_co9wDNSiK8zilHbH0xjDRHP8PkS1ZrgoM',
    '1jaR5XHmUoo6fYYw6E3sCl7lVuvXIbKZ1HwYssAxlFyE',
    '1xJKh8kgPX_WqfhOZ0fabv9VnlBcWMgKJoIpAo_GVYAA',
    # Real Zone 2
    '19iGURry9WwIM4F8zr2KbRHPdxNhnQuYFEJn8bZLaScU',
    '1h6za5W9a-plukPQiJko_dhLs1td0i5ZHjLf_XeIRusQ',
    '1cyRbeBMS8DI61yJJrlAt7NYI7mcwzWc_lwlOdvRuWYs',
    # Real Zone 3
    '1rf8HPssvBa49fybUmaYwP19PSIvUJwWEm3V_nsGPUlk',
    '11IZfbGKVNOEsk7ONL0Xj_VPPECqlB6ScJcEQ41Wy7Hw',
    '1Xzy4w-ju2Pq06CZHbrWHNW1gK-VS7PYavcm9AccjLAQ',
]

for file_id in tqdm(file_ids, desc="Downloading Files"):
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, quiet=False)

In [ ]:
import pandas as pd


file_paths = {
    "path_1_real": "/content/real_MISSA-KESWAL-01.xlsx",
    "path_1_synth": "/content/synth_MISSA-KESWAL-01.xlsx",
    "path_2_real": "/content/real_MISSA-KESWAL-02.xlsx",
    "path_2_synth": "/content/synth_MISSA-KESWAL-02.xlsx",
    "path_3_real": "/content/real_MISSA-KESWAL-03.xlsx",
    "path_3_synth": "/content/synth_MISSA-KESWAL-03.xlsx",
    "path_4_real": "/content/real_PINDORI-1.xlsx",
    "path_4_synth": "/content/synth_PINDORI-1.xlsx",
    "path_5_real": "/content/real_PINDORI-2.xlsx",
    "path_5_synth": "/content/synth_PINDORI-2.xlsx",
    "path_6_real": "/content/real_PINDORI-3.xlsx",
    "path_6_synth": "/content/synth_PINDORI-3.xlsx",
    "path_7_real": "/content/real_JOYAMAIR-4.xlsx",
    "path_7_synth": "/content/synth_JOYAMAIR-4.xlsx",
    "path_8_real": "/content/real_MINWAL-2.xlsx",
    "path_8_synth": "/content/synth_MINWAL-2.xlsx",
    "path_9_real": "/content/real_MINWAL-X-1.xlsx",
    "path_9_synth": "/content/synth_MINWAL-X-1.xlsx"
}



In [ ]:
# 2. Dictionary to store the actual DataFrames
dataframes = {}

print(f"{'Dataset Name':<25} | {'Row Count':<10}")
print("-" * 40)

# 3. Loop through, load, and count
for name, path in file_paths.items():
    try:
        df = pd.read_excel(path)

        # Store it in our dictionary so you can access it later (e.g., dataframes['path_1_real'])
        dataframes[name] = df

        # Print the row count
        print(f"{name:<25} | {len(df):<10}")

    except Exception as e:
        print(f"Error loading {name}: {e}")

In [ ]:
for name, df in dataframes.items():
    print(f"{name:<25} | {df.columns.tolist()}")

1. path_2 | 5627 rows
2. path_7 | 6081 rows
3. path_1 | 7019 rows
4. path_3 | 7542 rows
5. path_9 | 8169 rows
6. path_6 | 9463 rows
7. path_8 | 9878 rows
8. path_4 | 41707 rows
9. path_5 | 41995 rows

#4. Processing the data

In [ ]:
print(dataframes['path_7_real'].columns)

In [ ]:
mapping = {
    'RT': 'RES_DEEP',
    'RHOB': 'RHOB_combined',
    'RHOB_COMBINED': 'RHOB_combined',
}

# 1. First, rename the synth columns so they can be matched
synth_keys = [k for k in dataframes.keys() if '_synth' in k]
for key in synth_keys:
    dataframes[key] = dataframes[key].rename(columns=mapping)

# Rename in BOTH synth and real
for k in dataframes:
    dataframes[k] = dataframes[k].rename(columns=mapping)

# 2. Loop through each path number (1-9) to intersect columns
for i in range(1, 10):
    s_key = f'path_{i}_synth'
    r_key = f'path_{i}_real'

    # Check if both keys exist in your dictionary to avoid errors
    if s_key in dataframes and r_key in dataframes:
        # Find columns that exist in BOTH dataframes
        common_cols = dataframes[s_key].columns.intersection(dataframes[r_key].columns)

        # Filter both dataframes to only include these common columns
        dataframes[s_key] = dataframes[s_key][common_cols]
        dataframes[r_key] = dataframes[r_key][common_cols]

In [ ]:
# 3. Create your tuples (now they have identical shapes)
path_1_data = (dataframes['path_1_synth'], dataframes['path_1_real'])
path_2_data = (dataframes['path_2_synth'], dataframes['path_2_real'])
path_3_data = (dataframes['path_3_synth'], dataframes['path_3_real'])
path_4_data = (dataframes['path_4_synth'], dataframes['path_4_real'])
path_5_data = (dataframes['path_5_synth'], dataframes['path_5_real'])
path_6_data = (dataframes['path_6_synth'], dataframes['path_6_real'])
path_7_data = (dataframes['path_7_synth'], dataframes['path_7_real'])
path_8_data = (dataframes['path_8_synth'], dataframes['path_8_real'])
path_9_data = (dataframes['path_9_synth'], dataframes['path_9_real'])


PHYSICAL_BOUNDS = {
    "GR": (0, 200), "DT": (30, 180), "RHOB_combined": (1.2, 2.9),
    "RES_DEEP": (0.01, 10_000), "HP": (500, 15_000), "OB": (2_000, 40_000),
    "DT_NCT": (30, 180), "PPP": (0, 30_000),
}

SENTINEL_VALUES = {-999, -999.25, -999.0}


def clean_well_data(df, outlier_std=5, verbose=True, label=""):
    """
    Clean a single well DataFrame (synthetic OR real).
    1. Drop SPHI column if present.
    2. Replace sentinel values (-999, -999.25) with NaN.
    3. Replace values outside physical bounds with NaN.
    4. Replace extreme outliers (> outlier_std σ) with NaN.
    5. Drop rows with any remaining NaN in log columns.
    """
    df = df.copy()
    n_before = len(df)

    if "SPHI" in df.columns:
        df.drop(columns=["SPHI"], inplace=True)

    log_cols = [c for c in df.columns if c != "DEPTH"]

    for col in log_cols:
        df.loc[df[col].isin(SENTINEL_VALUES), col] = np.nan

    for col in log_cols:
        if col in PHYSICAL_BOUNDS:
            lo, hi = PHYSICAL_BOUNDS[col]
            df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan

    for col in log_cols:
        clean_vals = df[col].dropna()
        if len(clean_vals) < 10:
            continue
        mu, sig = clean_vals.mean(), clean_vals.std()
        if sig == 0:
            continue
        df.loc[(df[col] - mu).abs() > outlier_std * sig, col] = np.nan

    df.dropna(subset=log_cols, how='any', inplace=True)
    df.reset_index(drop=True, inplace=True)

    if verbose:
        removed = n_before - len(df)
        pct = 100 * removed / n_before if n_before else 0
        print(f"  [{label}]  {n_before} → {len(df)} rows  "
              f"(removed {removed}, {pct:.1f}%)")
    return df


def clean_all_wells(real_dfs, well_names, outlier_std=5):
    cleaned = []
    for name, df in zip(well_names, real_dfs):
        cleaned.append(clean_well_data(df, outlier_std=outlier_std,
                                       label=f"{name} REAL"))
    return cleaned


def clean_synthetic(df_syn, outlier_std=5, label="SYNTH"):
    return clean_well_data(df_syn, outlier_std=outlier_std, label=label)

In [ ]:
_all_path_tuples = {
    "path_1": path_1_data, "path_2": path_2_data, "path_3": path_3_data,
    "path_4": path_4_data, "path_5": path_5_data, "path_6": path_6_data,
    "path_7": path_7_data, "path_8": path_8_data, "path_9": path_9_data,
}

print("\n=== Cleaning all wells ===")
for _name, (_synth_df, _real_df) in _all_path_tuples.items():
    _cleaned_synth = clean_well_data(_synth_df, label=f"{_name} SYNTH")
    _cleaned_real  = clean_well_data(_real_df,  label=f"{_name} REAL")
    _all_path_tuples[_name] = (_cleaned_synth, _cleaned_real)

path_1_data = _all_path_tuples["path_1"]
path_2_data = _all_path_tuples["path_2"]
path_3_data = _all_path_tuples["path_3"]
path_4_data = _all_path_tuples["path_4"]
path_5_data = _all_path_tuples["path_5"]
path_6_data = _all_path_tuples["path_6"]
path_7_data = _all_path_tuples["path_7"]
path_8_data = _all_path_tuples["path_8"]
path_9_data = _all_path_tuples["path_9"]
print("=== Cleaning complete ===\n")

In [ ]:
order_of_dataset_processing = [
    path_2_data,
    path_7_data,
    path_1_data,
    path_3_data,
    path_9_data,
    path_6_data,
    path_8_data,
    path_4_data,
    path_5_data
]

In [ ]:
# Iterate through each tuple in your ordered list
for i, (synth_df, real_df) in enumerate(order_of_dataset_processing):
    print(f"--- Dataset Group {i+1} ---")

    # Print columns for the Synthetic DataFrame
    print(f"Synthetic Columns: {synth_df.columns.tolist()}")

    # Print columns for the Real DataFrame
    print(f"Real Columns:      {real_df.columns.tolist()}")
    print("\n")

In [ ]:
def processing_tuple(tuple_with_data):
    synth_df, real_df = tuple_with_data
    TARGET = 'PPP'
    EXCLUDE = ['DEPTH']
    features = [c for c in synth_df.columns if c != TARGET and c not in EXCLUDE]
    print(features)

    # TSTR: train/val from synthetic, blind test from real
    train_df, val_df = train_test_split(synth_df, test_size=0.3, random_state=42)

    # Combined synthetic (train + val) for final retraining
    combo = pd.concat([train_df, val_df], ignore_index=True)

    # --- Feature scaler (unchanged) ---
    scaler = StandardScaler()
    all_data = pd.concat([synth_df[features], real_df[features]], ignore_index=True)
    scaler.fit(synth_df[features].values)       # only synth

    X_all_s = scaler.transform(combo[features].values)
    X_test_s = scaler.transform(real_df[features].values)
    X_optuna_train = scaler.transform(train_df[features].values)
    X_optuna_val = scaler.transform(val_df[features].values)

    # --- Target scaler (NEW) ---
    target_scaler = StandardScaler()
    all_targets = pd.concat([synth_df[[TARGET]], real_df[[TARGET]]], ignore_index=True)
    target_scaler.fit(synth_df[[TARGET]].values) # only synth

    y_all = target_scaler.transform(combo[[TARGET]].values).ravel()
    y_test = target_scaler.transform(real_df[[TARGET]].values).ravel()
    y_optuna_train = target_scaler.transform(train_df[[TARGET]].values).ravel()
    y_optuna_val = target_scaler.transform(val_df[[TARGET]].values).ravel()

    return {
        'features': features,
        'scaler': scaler,
        'target_scaler': target_scaler,
        'X_all_s': X_all_s,
        'y_all': y_all,
        'X_test_s': X_test_s,
        'y_test': y_test,
        'X_optuna_train': X_optuna_train,
        'y_optuna_train': y_optuna_train,
        'X_optuna_val': X_optuna_val,
        'y_optuna_val': y_optuna_val,
    }

# 5. TRTR (5%, 10%, 20%) per path

In [ ]:
RIDGE_SPEC = {
    "name": "Ridge",
    "build": lambda p: Ridge(**p),
    "suggest": lambda t: {
        "alpha": t.suggest_float("alpha", 1e-4, 100.0, log=True),
    },
}

LIGHTGBM_SPEC = {
    "name": "LightGBM",
    "build": lambda p: lgb.LGBMRegressor(
        **p,
        random_state=42,
        verbosity=-1,
        device="gpu"  # ← GPU Enabled
    ),
    "suggest": lambda t: {
        "n_estimators":      t.suggest_int("n_estimators", 100, 2000),
        "learning_rate":     t.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth":         t.suggest_int("max_depth", 3, 12),
        "num_leaves":        t.suggest_int("num_leaves", 16, 256),
        "min_child_samples": t.suggest_int("min_child_samples", 5, 100),
        "subsample":         t.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         t.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        t.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    },
}

XGBOOST_SPEC = {
    "name": "XGBoost",
    "build": lambda p: xgb.XGBRegressor(
        **p,
        random_state=42,
        verbosity=0,
        tree_method="hist", # ← GPU Enabled
        device="cuda"           # ← GPU Enabled
    ),
    "suggest": lambda t: {
        "n_estimators":     t.suggest_int("n_estimators", 100, 2000),
        "learning_rate":    t.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth":        t.suggest_int("max_depth", 3, 12),
        "min_child_weight": t.suggest_float("min_child_weight", 1, 20),
        "subsample":        t.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":            t.suggest_float("gamma", 1e-8, 5.0, log=True),
        "reg_alpha":        t.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       t.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    },
}

RANDOM_FOREST_SPEC = {
    "name": "RandomForest",
    "build": lambda p: RandomForestRegressor(**p, random_state=42, n_jobs=-1),
    "suggest": lambda t: {
        "n_estimators":      t.suggest_int("n_estimators", 100, 1500),
        "max_depth":         t.suggest_int("max_depth", 4, 30),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf":  t.suggest_int("min_samples_leaf", 1, 20),
        "max_features":      t.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, 1.0]),
    },
}

CATBOOST_SPEC = {
    "name": "CatBoost",
    "build": lambda p: CatBoostRegressor(
        **p,
        random_seed=42,
        verbose=0,
        task_type="GPU" # ← GPU Enabled
    ),
    "suggest": lambda t: {
        "iterations":          t.suggest_int("iterations", 100, 2000),
        "learning_rate":       t.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "depth":               t.suggest_int("depth", 3, 10),
        "l2_leaf_reg":         t.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": t.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     t.suggest_float("random_strength", 1e-3, 10.0, log=True),
    },
}

KRR_SPEC = {
    "name": "KRR",
    "build": lambda p: KernelRidge(**p),
    "suggest": lambda t: {
        "kernel": t.suggest_categorical("kernel", ["rbf", "linear"]),
        "alpha":  t.suggest_float("alpha", 1e-4, 100.0, log=True),
        "gamma":  t.suggest_float("gamma", 1e-5, 1.0, log=True),
    },
}

In [ ]:
PATHS_SMALLEST_TO_BIGGEST = [
    ("path_2", path_2_data),
    ("path_7", path_7_data),
    ("path_1", path_1_data),
    ("path_3", path_3_data),
    ("path_9", path_9_data),
    ("path_6", path_6_data),
    ("path_8", path_8_data),
    ("path_4", path_4_data),
    ("path_5", path_5_data),
]

In [ ]:
all_reports = []
_t0_all, _m0_all = time.time(), _rss_mb()

Before running the code, making sure the GPUs for ML models will be used

In [ ]:
%reload_ext cuml.accel

## 5.0 TRTR to ensure the synthetic data has value

In [ ]:
TREE_SPECS = [RIDGE_SPEC, KRR_SPEC, LIGHTGBM_SPEC, XGBOOST_SPEC,
             RANDOM_FOREST_SPEC, CATBOOST_SPEC]

TRTR_FRACS = [0.05, 0.10, 0.20]

In [ ]:
_spec = _X_tr = _y_tr = _X_va = _y_va = None  # Optuna globals

In [ ]:
def _trtr_objective(trial):
    params = _spec["suggest"](trial)
    model  = _spec["build"](params)
    model.fit(_X_tr, _y_tr)
    return mean_squared_error(_y_va, model.predict(_X_va))

In [ ]:
def run_trtr_tuned(spec, data_tuple, dataset_name):
    """
    TRTR with fixed, spatially-disjoint real test set (same bottom-30% depth block
    used by pure-TSTR and semi-TSTR in the other notebook).
    Contiguous depth-block training, HP tuning on real-only with disjoint real val.
    """
    global _spec, _X_tr, _y_tr, _X_va, _y_va
    _, real_df = data_tuple
    TARGET, EXCLUDE = 'PPP', ['DEPTH']
    features = [c for c in real_df.columns if c not in [TARGET] + EXCLUDE]
    rows = []

    # ── Fixed spatial split (once per well, reused for all fracs & seeds) ──
    real_sorted = real_df.sort_values("DEPTH").reset_index(drop=True)
    n_real      = len(real_sorted)
    test_cut    = int(0.70 * n_real)
    real_pool   = real_sorted.iloc[:test_cut].reset_index(drop=True)   # train pool (top 70%)
    real_test   = real_sorted.iloc[test_cut:].reset_index(drop=True)   # FIXED test (bottom 30%)

    for frac in TRTR_FRACS:
        print(f"\n{'='*60}")
        print(f"  TRTR  {spec['name']}  —  {dataset_name}  (frac={frac:.0%})")
        print(f"{'='*60}")

        # ── Optuna: tune on contiguous block of real_pool, val = disjoint slice ──
        rng_hp       = np.random.RandomState(42)
        block_len_hp = max(1, int(frac * n_real))
        start_hp     = rng_hp.randint(0, max(1, len(real_pool) - block_len_hp))
        tr_hp        = real_pool.iloc[start_hp:start_hp + block_len_hp]
        va_hp        = real_pool.drop(tr_hp.index)                     # disjoint real val

        fs_hp = StandardScaler().fit(tr_hp[features])
        ts_hp = StandardScaler().fit(tr_hp[[TARGET]])

        _spec = spec
        _X_tr = fs_hp.transform(tr_hp[features].values)
        _y_tr = ts_hp.transform(tr_hp[[TARGET]].values).ravel()
        _X_va = fs_hp.transform(va_hp[features].values)
        _y_va = ts_hp.transform(va_hp[[TARGET]].values).ravel()

        study = optuna.create_study(direction="minimize")
        t0_optuna = time.time()
        study.optimize(_trtr_objective, n_trials=N_TRIALS, show_progress_bar=True)
        optuna_secs = time.time() - t0_optuna
        best_params = study.best_params
        print(f"  Optuna tuning took {optuna_secs:.2f} seconds")
        print(f"  Best params: {best_params}")

        # ── Multi-seed evaluation with tuned params ─────────────────────
        seed_r2, seed_rmse, seed_mae = [], [], []
        seed_fit_times, seed_predict_times = [], []

        for seed in tqdm(SEEDS, desc=f"  {spec['name']} | {dataset_name} | TRTR {frac:.0%}"):
            # Contiguous depth-block sample from real_pool (NOT random rows)
            rng        = np.random.RandomState(seed)
            block_len  = max(1, int(frac * n_real))
            max_start  = max(1, len(real_pool) - block_len)
            start      = rng.randint(0, max_start)
            real_train = real_pool.iloc[start:start + block_len]

            fs = StandardScaler().fit(real_train[features])
            ts = StandardScaler().fit(real_train[[TARGET]])

            X_tr = fs.transform(real_train[features].values)
            y_tr = ts.transform(real_train[[TARGET]].values).ravel()
            X_te = fs.transform(real_test[features].values)               # FIXED test

            model = spec["build"](best_params)
            t0_fit = time.time()
            model.fit(X_tr, y_tr)
            fit_time = time.time() - t0_fit

            t0_predict = time.time()
            preds = ts.inverse_transform(model.predict(X_te).reshape(-1, 1)).ravel()
            predict_time = time.time() - t0_predict

            seed_fit_times.append(fit_time)
            seed_predict_times.append(predict_time)

            truth = real_test[TARGET].values
            seed_r2.append(r2_score(truth, preds))
            seed_rmse.append(np.sqrt(mean_squared_error(truth, preds)))
            seed_mae.append(mean_absolute_error(truth, preds))

        seed_r2, seed_rmse, seed_mae = np.array(seed_r2), np.array(seed_rmse), np.array(seed_mae)
        seed_fit_times = np.array(seed_fit_times)
        seed_predict_times = np.array(seed_predict_times)

        n = len(SEEDS)
        total_fit_time     = seed_fit_times.sum()
        total_predict_time = seed_predict_times.sum()

        print(f"  R²   = {seed_r2.mean():.4f} ± {seed_r2.std():.4f}  "
              f"[{seed_r2.min():.4f}, {seed_r2.max():.4f}]")
        print(f"  RMSE = {seed_rmse.mean():.4f} ± {seed_rmse.std():.4f}")
        print(f"  MAE  = {seed_mae.mean():.4f} ± {seed_mae.std():.4f}")
        print(f"  Avg Fit Time = {seed_fit_times.mean():.4f} ± {seed_fit_times.std():.4f} s")
        print(f"  Total Fit Time = {total_fit_time:.4f} s")
        print(f"  Avg Predict Time = {seed_predict_times.mean():.4f} ± {seed_predict_times.std():.4f} s")
        print(f"  Total Predict Time = {total_predict_time:.4f} s")

        rows.append({
            "well": dataset_name, "model": spec["name"], "mode": "TRTR",
            "frac": frac, "best_params": best_params,
            "r2_mean": float(seed_r2.mean()), "r2_std": float(seed_r2.std()),
            "r2_ci95": float(1.96 * seed_r2.std() / np.sqrt(n)),
            "rmse_mean": float(seed_rmse.mean()), "rmse_std": float(seed_rmse.std()),
            "mae_mean": float(seed_mae.mean()), "mae_std": float(seed_mae.std()),
            "optuna_secs": optuna_secs,
            "fit_time_mean": float(seed_fit_times.mean()), "fit_time_std": float(seed_fit_times.std()),
            "total_fit_time": float(total_fit_time),
            "predict_time_mean": float(seed_predict_times.mean()), "predict_time_std": float(seed_predict_times.std()),
            "total_predict_time": float(total_predict_time),
            "r2_all_seeds": seed_r2.tolist(),
        })
    return rows

In [ ]:
trtr_rows = []
for path_name, data in PATHS_SMALLEST_TO_BIGGEST:
    print(f"\n{'#'*60}\n#  TRTR CHECK: {path_name}\n{'#'*60}")
    for spec in TREE_SPECS:
        trtr_rows += run_trtr_tuned(spec, data, path_name)
        pd.DataFrame(trtr_rows).to_csv(os.path.join(SAVE_DIR, "trtr_regularity_tuned.csv"), index=False)  # ← ADD THIS LINE

trtr_df = pd.DataFrame(trtr_rows)
print(f"\n>>> Saved trtr_regularity_tuned.csv ({len(trtr_df)} rows)")
print(trtr_df.drop(columns=["best_params","r2_all_seeds"]).to_string(index=False, float_format="%.4f"))

In [ ]:
raise ValueError("Stop here")